In [ ]:
# ──────────────────────────────────────────────────────────────
# main.ipynb와 통신
# 모델 로딩은 서버 시작 시 1회만 이루어진다
# port 8001 
# ──────────────────────────────────────────────────────────────

In [ ]:
# # ──────────────────────────────────────────────────────────────
# # ocr 관련 패키지 설치 (최초 1회)
# # torch 2.7.0+cu118이 이미 설치된 환경 기준
# # --extra-index-url 추가로 기존 torch를 건드리지 않고 설치
# # ──────────────────────────────────────────────────────────────
# import subprocess

# subprocess.run([
#     'pip', 'install', '-q',
#     'chandra-ocr[hf]',
#     'transformers', 'accelerate', 'pillow',
#     'fastapi', 'uvicorn', 'python-multipart',
#     '--extra-index-url', 'https://download.pytorch.org/whl/cu118'
# ])

In [ ]:
# ──────────────────────────────────────────────────────────────
# 필요 라이브러리 import
# ──────────────────────────────────────────────────────────────
import torch
import io
import os
import base64
import threading
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig, AutoModelForCausalLM
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse
import uvicorn
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(dotenv_path="..\..\dataset\config\.env")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Chandra 전용 라이브러리 (pip install chandra-ocr[hf] 필요)
# ──────────────────────────────────────────────────────────────
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown

In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP1: 모델 로드 및 전역 설정
# GPU 사용 가능 여부 확인
# ──────────────────────────────────────────────────────────────
deviceType = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {deviceType}')

if deviceType == 'cuda':
    vramGB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {vramGB:.1f} GB')
else:
    print(' GPU를 찾을 수 없습니다. CPU로 실행되며 속도가 느릴 수 있습니다.')

In [ ]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version (torch):", torch.version.cuda)
print("cudnn version:", torch.backends.cudnn.version())

if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))

In [ ]:
## 현재 내 컴의 CUDA와 torch 버전이 일치하지 않아 계속 오류가 발생함;ㅠㅠ
## 기존 torch 삭제하고 새 torch를 설치하자
#!pip uninstall torch torchvision torchaudio -y

In [ ]:
#!pip uninstall torch -y

In [ ]:
## 내 CUDA와 호환되는 버전의 torch 설치
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
#!nvidia-smi

In [ ]:
# ──────────────────────────────────────────────────────────────
# 부동소수점 타입 결정
# ──────────────────────────────────────────────────────────────
if torch.cuda.get_device_capability()[0] >= 8:
    # 고속 attention 메커니즘을 구현하는 라이브러리
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

In [ ]:
# 환경변수에서 토큰 가져오기
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    # notebook_login(창 띄우기)을 방지하기 위해 직접 token을 인자로 전달합니다.
    login(token=hf_token)
    print("✅ 허깅페이스 로그인 성공!")
else:
    print("❌ HF_TOKEN을 찾을 수 없습니다. .env 파일의 변수명을 확인해주세요.")

In [ ]:
# !pip install -U bitsandbytes>=0.46.1

In [ ]:
# ──────────────────────────────────────────────────────────────
# 모델 로드 (서버 시작 시 1회 실행) — 4bit NF4 양자화 적용
# ──────────────────────────────────────────────────────────────
MODEL_ID = 'datalab-to/chandra-ocr-2'
# MODEL_ID="fredrezones55/chandra-ocr-2"

quantConfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=False
)

print(f'모델 로딩 중: {MODEL_ID}')

model = AutoModelForImageTextToText.from_pretrained(
MODEL_ID,
    quantization_config=quantConfig,
    attn_implementation="sdpa",
    device_map='auto',
)
model.eval()
model.processor = AutoProcessor.from_pretrained(MODEL_ID)
model.processor.tokenizer.padding_side = 'left'

print('모델 로드 완료!')

if deviceType == 'cuda':
    usedVram = torch.cuda.memory_allocated(0) / (1024 ** 3)
    print(f'현재 VRAM 사용량: {usedVram:.2f} GB')

In [ ]:
import re

def cleanOcrText(raw_text):
    # 1. ```markdown 이나 ``` 같은 코드 블록 표시 제거
    text = re.sub(r'```(?:markdown|html|txt)?', '', raw_text)
    text = text.replace('```', '')
    
    # 2. 앞뒤 공백 제거
    text = text.strip()
    
    # 3. (선택) 만약 "Here is the text:" 같은 특정 패턴이 반복된다면 제거
    text = re.sub(r'^(Sure|Here|The|Based on).*?:\n?', '', text, flags=re.IGNORECASE)
    
    return text.strip()

In [ ]:
# ──────────────────────────────────────────────────────────────
# OCR 추론 함수 — 공식 chandra-ocr 패키지 사용
# prompt_type: 'ocr' (빠름, 텍스트만) / 'ocr_layout' (느림, 마크다운 레이아웃)
# ──────────────────────────────────────────────────────────────

MAX_IMAGE_SIZE = 1024  # 긴 변 기준 최대 픽셀

def resizeImage(pilImage: Image.Image) -> Image.Image:
    """ 이미지가 너무 크면 비율 유지하며 축소 """
    w, h = pilImage.size
    if max(w, h) > MAX_IMAGE_SIZE:
        ratio = MAX_IMAGE_SIZE / max(w, h)
        pilImage = pilImage.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    return pilImage

def runOcr(pilImage: Image.Image, promptType: str = 'ocr') -> str:
    """ PIL 이미지를 받아 Chandra OCR 2로 텍스트를 추출하여 반환 """
    if pilImage.mode != 'RGB':
        pilImage = pilImage.convert('RGB')

    pilImage = resizeImage(pilImage)
    print(f'처리 이미지 크기: {pilImage.size}, 모드: {promptType}')

    custom_prompt = (
        """Perform OCR on this image. 
        Output ONLY the recognized text. 
        Do not include any greeting, explanation, or markdown code blocks like ```.""" ## 제대로 안먹는 중;ㅋㅋ
    )

    batch = [
        BatchInputItem(
            image=pilImage,
            prompt=custom_prompt, ## 대화형식의 답변이 아닌 오직 텍스트 출력만을 위해 추가
            prompt_type=promptType
        )
    ]

    result = generate_hf(batch, model)[0]
    return cleanOcrText(parse_markdown(result.raw))


def bytesToPilImage(imageBytes: bytes) -> Image.Image:
    """ 바이트 데이터를 PIL Image 객체로 변환 """
    return Image.open(io.BytesIO(imageBytes))

In [ ]:
# ──────────────────────────────────────────────────────────────
# ocr 엔드포인트 설정
# ──────────────────────────────────────────────────────────────
app = FastAPI(title="Chandra OCR Service")


@app.post("/ocr")
async def extract_text(image: UploadFile = File(...)):
    try:
        # 1. 업로드된 파일 읽기
        content = await image.read()
        
        # 2. 바이트 데이터를 PIL 이미지로 변환
        pil_img = Image.open(io.BytesIO(content))
        
        # 3. 분리된 추론 함수 호출 (내부에서 리사이징 등 처리)
        ocr_result = runOcr(pil_img)

        print("ocr_result : "+ocr_result)

        return {
            "success": True,
            "extracted_text":ocr_result,
            "model" : MODEL_ID
        }

    except Exception as e:
        print(f"Error: {str(e)}")
        return JSONResponse(
            status_code=500,
            content={"success": False, "error": str(e)}
        )

In [ ]:
print(f"model : {MODEL_ID}")

In [ ]:
def run_server():
    # .env에서 포트를 가져오거나 기본값 8001 사용
    port = int(os.getenv("OCR_PORT", 8001))
    # 노트북 충돌 방지를 위해 uvicorn.run 사용
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")

# 이미 서버가 돌고 있는지 확인 (재실행 시 에러 방지용)
# 주피터에서 셀을 여러 번 누를 때를 대비해 thread가 살아있는지 체크하면 좋습니다.
ocr_thread = threading.Thread(target=run_server, daemon=True)
ocr_thread.start()

print(f"🚀 OCR 서버가 백그라운드에서 시작되었습니다!")
print(f"문서 주소: http://localhost:{os.getenv('OCR_PORT', 8001)}/docs")

In [ ]:
# ──────────────────────────────────────────────────────────────
# 간단한 테스트 예시 (로컬 이미지 파일로 바로 테스트)
# ──────────────────────────────────────────────────────────────
import os

# 테스트할 이미지 경로 (본인 이미지로 변경)
testImagePath = '../../dataset/test.jpg'

if os.path.exists(testImagePath):
    testImage = Image.open(testImagePath)
    print('OCR 실행 중...')
    extractedText = runOcr(testImage)
    print('=' * 50)
    print('추출된 텍스트:')
    print(extractedText)
    print('=' * 50)
else:
    print(f'테스트 이미지({testImagePath})가 없습니다.')
    print('같은 폴더에 test.jpg를 넣고 다시 실행하거나,')
    print('아래 API 서버를 실행해서 /ocr 엔드포인트로 테스트하세요.')